In [1]:
# Set workspace
import sys
import os
workspace_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Libraries

In [2]:
# Libraries
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import gpxpy

In [3]:
# Other helpers

from helper_functions.gpx_to_tabular import (
    gpx_to_tabular,
)

from helper_functions.sector_data_cleanup import (
    clean_runner_race_sector_data,
    apply_rest_only_checkpoint_correction,
    add_sector_distance_features,
    build_sector_boundaries,
    build_course_sector_elevation_aggregation,
    add_sector_elevation_features,
)

---
# Get data

In [4]:
# Config
years = [2022, 2023, 2024, 2025]

Load data

In [5]:
# Get local race results data for UTMB

# Dictionary | key = year
utmb_race_results_df_dict = dict()
for year in years:

    utmb_year_race_results_df = pd.read_parquet(
        os.path.join("..", "utmb_under_27h/data", f"utmb_{year}_race_results.parquet")
    )
    utmb_year_race_results_df = utmb_year_race_results_df.assign(year=year)

    print(f"Data for year {year} fetched and loaded successfully")
    utmb_race_results_df_dict[year] = utmb_year_race_results_df

# Single dataframe with all years
utmb_race_results_df = pd.concat(
    utmb_race_results_df_dict.values(),
    ignore_index=True
).reset_index(drop=True)

Data for year 2022 fetched and loaded successfully
Data for year 2023 fetched and loaded successfully
Data for year 2024 fetched and loaded successfully
Data for year 2025 fetched and loaded successfully


In [6]:
# Get sector data for UTMB

# Dictionary | key = year
utmb_sector_data_dict = dict()
for year in years:

    with open(os.path.join(f"data", f"utmb_{year}_sector_data.pkl"), "rb") as f:
        utmb_year_sector_data_dict = pickle.load(f)

    print(f"Data for year {year} fetched and loaded successfully")
    utmb_sector_data_dict[year] = utmb_year_sector_data_dict

Data for year 2022 fetched and loaded successfully
Data for year 2023 fetched and loaded successfully
Data for year 2024 fetched and loaded successfully
Data for year 2025 fetched and loaded successfully


In [7]:
# UTMB GPX - We use the 2025 GPX file as the reference course for all years. This assumes the course geometry is close enough across years for descriptive elevation/distance features.
gpx_file_path = "utmb_2025_course.gpx"
with open(gpx_file_path, "r") as f:
    utmb_gpx = gpxpy.parse(f)

utmb_course_df = gpx_to_tabular(utmb_gpx)

Connect Race data & Sector data

In [8]:
# Connect Race data & Sector data & Save locally (/Load)

CONNECT_RACE_SECTOR_DATA = False
SAVE_RACE_SECTOR_DATA = False

runner_race_sector_data_dict = dict()
runner_race_sector_data_year_dfs = []

# Connect
if CONNECT_RACE_SECTOR_DATA:
    for year in years:
        print(" - Connecting year {} ...".format(year))

        # Year-level race results table
        utmb_race_results_year_df = (
            utmb_race_results_df_dict[year]
            .set_index("runner_bib_number")
        )

        # Year-level sector data dict: {bib_number -> sector dataframe}
        utmb_sector_data_year_dict = utmb_sector_data_dict[year]

        # Collect runner-level sector data for this year
        runner_race_sector_data_year_list = []
        for bib_number, utmb_sector_data_year_bib_df in utmb_sector_data_year_dict.items():

            if bib_number not in utmb_race_results_year_df.index:
                continue

            utmb_race_results_year_bib_series = utmb_race_results_year_df.loc[bib_number]

            utmb_sector_data_year_bib_df = (
                utmb_sector_data_year_bib_df
                .assign(**utmb_race_results_year_bib_series.to_dict())
            )

            # Store runner-level dataframe in the dict and in the year list
            runner_race_sector_data_dict[(year, bib_number)] = utmb_sector_data_year_bib_df
            runner_race_sector_data_year_list.append(utmb_sector_data_year_bib_df)

        # Drop all-NA columns once, after the year loop
        runner_race_sector_data_year_list = [
            df.dropna(axis=1, how="all")
            for df in runner_race_sector_data_year_list
            if df is not None and not df.empty
        ]

        # Combine all runners for the year into one dataframe
        if runner_race_sector_data_year_list:
            runner_race_sector_data_year_df = pd.concat(
                runner_race_sector_data_year_list,
                ignore_index = True,
            )
            runner_race_sector_data_year_dfs.append(runner_race_sector_data_year_df)

            # Save
            if SAVE_RACE_SECTOR_DATA:
                save_path = os.path.join(
                    "..",
                    "pacing_strategy",
                    "data",
                    f"runner_race_sector_data_{year}.parquet",
                )
                runner_race_sector_data_year_df.to_parquet(save_path, index = False)
                print(" - Saved year {} to {}".format(year, save_path))

# Load
if not CONNECT_RACE_SECTOR_DATA:
    for year in years:
        load_path = os.path.join("..", "pacing_strategy", "data", f"runner_race_sector_data_{year}.parquet")
        runner_race_sector_data_year_df = pd.read_parquet(load_path)

        runner_race_sector_data_year_dfs.append(runner_race_sector_data_year_df)
        print(" - Loaded year {} from {}".format(year, load_path))

# Combine all years
runner_race_sector_data_df = pd.concat(
    runner_race_sector_data_year_dfs,
    ignore_index = True,
)

 - Loaded year 2022 from ..\pacing_strategy\data\runner_race_sector_data_2022.parquet
 - Loaded year 2023 from ..\pacing_strategy\data\runner_race_sector_data_2023.parquet
 - Loaded year 2024 from ..\pacing_strategy\data\runner_race_sector_data_2024.parquet
 - Loaded year 2025 from ..\pacing_strategy\data\runner_race_sector_data_2025.parquet


--- 
# About data & Data work

In [9]:
# Assign Runner ID -  Runner/Race/Year unique identifier
runner_race_sector_data_df = runner_race_sector_data_df.assign(
    runner_id = lambda x: x.groupby([
        "runner_bib_number",
        "race_id",
        "year",
    ]).ngroup() + 1
)

About data

In [10]:
# About data
print("About data:")
print("-------------------------------------")
print("Race IDs: {}".format(runner_race_sector_data_df["race_id"].unique()))
print("Race names: {}".format(runner_race_sector_data_df["race_name"].unique()))
print("Years: {}".format(runner_race_sector_data_df["year"].unique()))
print("Number of unique runners: {:,}".format(runner_race_sector_data_df["runner_url"].nunique()))
print("Number of unique Runner/Race/Year combinations: {:,}".format(runner_race_sector_data_df["runner_id"].nunique()))

print("\nRunners per year:")
for year, n_runners in runner_race_sector_data_df.groupby("year")["runner_id"].nunique().items():
    print(" - year {} - runners {:,}".format(year, n_runners))

About data:
-------------------------------------
Race IDs: ['utmb']
Race names: ['UTMB®']
Years: [2022 2023 2024 2025]
Number of unique runners: 9,528
Number of unique Runner/Race/Year combinations: 10,563

Runners per year:
 - year 2022 - runners 2,627
 - year 2023 - runners 2,687
 - year 2024 - runners 2,760
 - year 2025 - runners 2,489


In [11]:
# About Sectors
print("About sectors:")
print("-------------------------------------")
print("Number of sector checpoints: {:,}".format(runner_race_sector_data_df["sector_checkpoint_id"].nunique()))
print("Sector checkpoint IDs: {}".format(runner_race_sector_data_df["sector_checkpoint_id"].unique().tolist()))

print("Do all races/years have the same number of sectors? {}".format(
    runner_race_sector_data_df.groupby(["race_id", "year"])["sector_checkpoint_id"].nunique().nunique() == 1
))

sector_sets = runner_race_sector_data_df.groupby(["race_id", "year"])["sector_checkpoint_id"].apply(lambda s: frozenset(s.unique()))
print("Do all races/years have the same sector IDs? {}".format(
    sector_sets.nunique() == 1
))

expected_sector_count = runner_race_sector_data_df.groupby(["race_id", "year"])["sector_checkpoint_id"].nunique()
runner_sector_counts = runner_race_sector_data_df.groupby(["race_id", "year", "runner_bib_number"])["sector_checkpoint_id"].nunique()
print("Do all runners have all of the sectors? {}".format(
    runner_sector_counts.groupby(["race_id", "year"]).min().eq(expected_sector_count).all()
))

About sectors:
-------------------------------------
Number of sector checpoints: 31
Sector checkpoint IDs: [0, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 18, 20, 22, 24, 27, 29, 30, 31, 33, 34, 35, 43, 44, 143, 19, 144, 9, 23]
Do all races/years have the same number of sectors? False
Do all races/years have the same sector IDs? False
Do all runners have all of the sectors? True


In [12]:
# About Sectors - Deep
sector_data_summary = (
    runner_race_sector_data_df
    .groupby("year")
    .aggregate(
        number_of_sectors = ("sector_checkpoint_id", "nunique"),
        sector_checkpoint_ids = ("sector_checkpoint_id", lambda x: [int(value) for value in sorted(x.unique())]),
    )
    .reset_index()
)

for _, row in sector_data_summary.iterrows():
    print(f"\nYear: {row['year']}")
    print("----------------------------------------")
    print(f"Number of sectors: {row['number_of_sectors']}")
    print(f"Sector checkpoint IDs: {row['sector_checkpoint_ids']}")

# Common and Unique sectors by years
sector_ids_by_year = (
    runner_race_sector_data_df
    .assign(year = lambda df: df["year"].astype(int))
    .groupby("year")["sector_checkpoint_id"]
    .apply(lambda s: set(int(x) for x in s.unique()))
)

all_sector_ids = set.union(*sector_ids_by_year.tolist())
common_sector_ids = set.intersection(*sector_ids_by_year.tolist())
not_common_sector_ids = all_sector_ids - common_sector_ids

all_years = sorted(sector_data_summary["year"].astype(int).unique().tolist())
sector_years = (
    runner_race_sector_data_df
    .groupby("sector_checkpoint_id")["year"]
    .apply(lambda s: sorted(s.astype(int).unique().tolist()))
)

print("\nAbout common sectors:")
print("----------------------------------------")
print("Number of common sector IDs: {} ({:.0f}%)".format(len(common_sector_ids), len(common_sector_ids) / len(all_sector_ids) * 100))
print("Number of not common sector IDs: {} ({:.0f}%)".format(len(not_common_sector_ids), len(not_common_sector_ids) / len(all_sector_ids) * 100))
print("Common sector IDs across all years: {}".format(sorted(common_sector_ids)))
print("Sector IDs not common to all years, with years present:")
for sector_id in sorted(not_common_sector_ids):
    missing_years = sorted(set(all_years) - set(sector_years[sector_id]))
    print(" - {} -> missing years: {}".format(sector_id, missing_years))


Year: 2022
----------------------------------------
Number of sectors: 27
Sector checkpoint IDs: [0, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 18, 20, 22, 24, 27, 29, 30, 31, 33, 34, 35, 43, 44, 143]

Year: 2023
----------------------------------------
Number of sectors: 26
Sector checkpoint IDs: [0, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 18, 19, 20, 22, 24, 27, 30, 31, 33, 34, 35, 44, 144]

Year: 2024
----------------------------------------
Number of sectors: 28
Sector checkpoint IDs: [0, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 18, 19, 20, 22, 23, 24, 27, 30, 31, 33, 34, 35, 44, 144]

Year: 2025
----------------------------------------
Number of sectors: 26
Sector checkpoint IDs: [0, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 18, 19, 20, 22, 24, 27, 30, 31, 33, 34, 35, 44, 144]



About common sectors:
----------------------------------------
Number of common sector IDs: 23 (74%)
Number of not common sector IDs: 8 (26%)
Common sector IDs across all years: [0, 2, 3, 4, 6, 7, 8, 11, 12, 13, 14, 15, 18, 20, 22, 24, 27, 30, 31, 33, 34, 35, 44]
Sector IDs not common to all years, with years present:
 - 5 -> missing years: [2025]
 - 9 -> missing years: [2022, 2023]
 - 19 -> missing years: [2022]
 - 23 -> missing years: [2022, 2023, 2025]
 - 29 -> missing years: [2023, 2024, 2025]
 - 43 -> missing years: [2023, 2024, 2025]
 - 143 -> missing years: [2023, 2024, 2025]
 - 144 -> missing years: [2022]


Work on sector data - cleanup and new features
- Add some additional sector description columns
- Correct rest only sectors
- Sector absolute and relative distance
- Calculate elevation features with help of GPX data

In [13]:
# Raw Sector data display 
sector_cols = runner_race_sector_data_df.columns[
    runner_race_sector_data_df.columns.str.contains("sector")
]

display(
    runner_race_sector_data_df
    .loc[:, ["runner_id"] + sector_cols.tolist()]
    .head(30)
)

,runner_id,sector_checkpoint_id,sector_arrival_time,sector_departure_time,sector_rest_time_seconds,sector_speed_kmh,sector_pace_min_per_km,sector_time_seconds,sector_time,sector_race_time_at_checkpoint,sector_checkpoint_rank,sector_checkpoint_rank_change,sector_rest_time,sector_predicted_time
0,1,0,None,2022-08-26T15:59:57.000Z,NaN,NaN,NaN,NaN,None,None,NaN,NaN,None,NaN
1,1,2,2022-08-26T17:08:12.160Z,2022-08-26T17:08:12.160Z,NaN,11.70,5.13,4095.16,1:08:15,1:08:15,2.0,NaN,None,NaN
2,1,3,2022-08-26T17:46:37.540Z,2022-08-26T17:46:37.540Z,NaN,13.07,4.59,6400.54,0:38:25,1:46:41,3.0,1.0,None,NaN
3,1,4,2022-08-26T18:37:17.930Z,2022-08-26T18:37:17.930Z,NaN,11.43,5.25,9440.93,0:50:40,2:37:21,2.0,-1.0,None,NaN
4,1,5,2022-08-26T19:30:25.280Z,2022-08-26T19:30:25.280Z,NaN,9.41,6.38,12628.28,0:53:07,3:30:28,1.0,-1.0,None,NaN
5,1,6,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,None,NaN
6,1,7,2022-08-26T20:52:38.090Z,2022-08-26T20:55:24.650Z,166.56,7.75,7.74,17561.09,1:22:13,4:52:41,2.0,1.0,0:02:47,NaN
7,1,8,2022-08-26T22:14:01.000Z,2022-08-26T22:14:01.000Z,NaN,8.12,7.39,22444.00,1:18:36,6:14:04,3.0,1.0,None,NaN
8,1,11,2022-08-26T22:59:53.820Z,2022-08-26T22:59:53.820Z,NaN,9.35,6.42,25196.82,0:45:53,6:59:57,2.0,-1.0,None,NaN
9,1,12,2022-08-26T23:34:24.000Z,2022-08-26T23:34:24.000Z,NaN,6.94,8.65,27267.00,0:34:30,7:34:27,2.0,0.0,None,NaN


In [14]:
# Work on sector data - pipe with helper_functions/sector_data_cleanup.py

runner_race_sector_data_df = (
    runner_race_sector_data_df
    .pipe(clean_runner_race_sector_data) # Add some additional sector description columns
    .pipe(apply_rest_only_checkpoint_correction) # Correct rest only sectors
    .pipe(add_sector_distance_features) # Sector absolute and relative distance
)

# - Build sector boundaries - for connecting with GPX data
# - Aggregate GPX course features by sector
# - Add elevation features to sector data and calculate additional elevation sector features
sector_boundaries_df = build_sector_boundaries(
    sector_df = runner_race_sector_data_df
)
course_sector_elevation_df = build_course_sector_elevation_aggregation(
    utmb_course_df = utmb_course_df,
    sector_boundaries_df = sector_boundaries_df
)
runner_race_sector_data_df = add_sector_elevation_features(
    sector_df = runner_race_sector_data_df,
    course_sector_agg_df = course_sector_elevation_df,
)

In [15]:
# Sector IDs: 1-max, not "random" values (Race/Year specific)
runner_race_sector_data_df = (
    runner_race_sector_data_df
    .assign(
        sector_number = lambda x: x
        .groupby(["race_id", "year"])["sector_checkpoint_id"]
        .rank(method="dense", ascending=True)
        .astype(int)
    )
)

# Sanity check
check_df = (
    runner_race_sector_data_df[["race_id", "year", "sector_checkpoint_id", "sector_number"]]
    .drop_duplicates()
)

assert (
    check_df.groupby(["race_id", "year"])["sector_checkpoint_id"].nunique()
    == check_df.groupby(["race_id", "year"])["sector_number"].nunique()
).all(), "Sector checkpoint mapping is not 1-to-1 within at least one race/year."

assert (
    check_df.groupby(["race_id", "year"])["sector_number"].min() == 1
).all(), "Sector checkpoint numbering does not start at 1 for at least one race/year."

Analysis data - Finishers

In [16]:
# Split finishers & DNFs
runner_race_sector_data_dnf_df = runner_race_sector_data_df.loc[
    ~runner_race_sector_data_df["runner_is_finisher"]
    ].reset_index(drop=True).copy()

runner_race_sector_data_finisher_df = runner_race_sector_data_df.loc[
    runner_race_sector_data_df["runner_is_finisher"]
    ].reset_index(drop=True).copy()

In [17]:
# Pacing analysis data = finishers
pacing_analysis_data = runner_race_sector_data_finisher_df.copy()

---
# Race time normalization

- Raw race times are not directly comparable because runners have different pre-race ability, and different years may have different conditions.
- We estimate each runner's expected race time from pre-race ability and race year.
- Normalization lets us compare runners more fairly.
- We can estimate whether the runner performed better or worse than expected.

Estimate expected runner race time (Linear model)
- Based on UTMB index & Race year
- T ~ UTMB_index + Race_year + UTMB_index:Race_year

In [21]:
# Race time normalization data - one row per runner/year
df_race_time_normalization = (
    pacing_analysis_data
    .drop_duplicates(subset=["runner_id"])
    [["runner_id", "runner_race_time_hours", "runner_overall_index", "year"]]
    .dropna(how="any")
    .reset_index(drop=True)
    .copy()
)

In [22]:
# Linear model - Fit & Predict

# Fit 
model_race_time_normalization = smf.ols(
    formula = "runner_race_time_hours ~ runner_overall_index * C(year)",
    data = df_race_time_normalization,
)
fit_race_time_normalization = model_race_time_normalization.fit()

# Predict & store predictions
y_pred_race_time_normalization = fit_race_time_normalization.predict(df_race_time_normalization)
df_race_time_normalization["runner_race_time_hours_pred"] = y_pred_race_time_normalization

In [23]:
# Coefficients
coef_race_time_normalization_df = (
    pd.DataFrame({
        "coef": fit_race_time_normalization.params,
        "std_err": fit_race_time_normalization.bse,
        "p_value": fit_race_time_normalization.pvalues,
        "ci_lower": fit_race_time_normalization.conf_int()[0],
        "ci_upper": fit_race_time_normalization.conf_int()[1],
    })
    .reset_index(names="term")
)

coef_race_time_normalization_df

,term,coef,std_err,p_value,ci_lower,ci_upper
0,Intercept,66.835956,0.472696,0.000000,65.909325,67.762586
1,C(year)[T.2023],1.531827,0.673181,0.022906,0.212182,2.851471
2,C(year)[T.2024],2.794643,0.674833,0.000035,1.471760,4.117525
3,C(year)[T.2025],2.828759,0.672894,0.000027,1.509679,4.147839
4,runner_overall_index,-0.048735,0.000814,0.000000,-0.050331,-0.047139
5,runner_overall_index:C(year)[T.2023],-0.002351,0.001158,0.042417,-0.004621,-0.000080
6,runner_overall_index:C(year)[T.2024],-0.004656,0.001162,0.000062,-0.006934,-0.002379
7,runner_overall_index:C(year)[T.2025],-0.005087,0.001169,0.000014,-0.007378,-0.002796


In [24]:
# Calculate absolute race time residual: Expected - Actual
df_race_time_normalization = (
    df_race_time_normalization
    .assign(
        runner_race_time_hours_residual_abs = lambda df: (
            df["runner_race_time_hours_pred"] - df["runner_race_time_hours"]
        )
    )
)

---
# Sector time normalization

- Raw sector times are not directly comparable because runners have different pre-race ability and sectors differ in distance, elevation, terrain.
- Normalization lets us compare runners sector times better.
- We normalize sector times so we can compare how runners distributed their race effort across sectors, relative to what would be expected for their ability, year, and sector.

---
# Pacing analysis

- Which pacing patterns are associated with better (normalized) race performance.